# EDA — Perfilado de datos crudos (The Playbook)

Demostración en vivo del **Paso 1 (Entorno / Datos Iniciales)** de la defensa EC1: el estado de los datos *antes* de cualquier procesamiento.

**Importante:** `partidos_demo.csv` es un dataset **sintético** (inventado por el equipo para esta demo), con la misma forma exacta que un CSV real de Football-Data.co.uk. Tiene huecos e inconsistencias de nombre **a propósito**, para que el EDA tenga algo real que encontrar. Antes de la entrega final hay que repetir este mismo notebook con el CSV real descargado de Football-Data.co.uk (basta con cambiar el nombre de archivo en la celda de carga).

Este notebook queda **deliberadamente incompleto** al final — la idea es agregar 1-2 exploraciones más en vivo el día de la defensa, no leer un resultado ya cerrado.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4)

## Carga de datos

Elegir **una** de las dos opciones según dónde se corra el notebook.

In [ ]:
# Opción A — corriendo el notebook desde el repo clonado (Jupyter local):
df = pd.read_csv("partidos_demo.csv")

# Opción B — en Google Colab, si subiste el CSV suelto en vez de clonar el repo:
# from google.colab import files
# subido = files.upload()  # elegir partidos_demo.csv en el diálogo
# df = pd.read_csv(list(subido.keys())[0])

df.shape

## 1. Estado antes de cualquier procesamiento

Volumen, tipos de dato y una muestra cruda — todavía sin transformar nada.

In [ ]:
print(f"Filas: {df.shape[0]}, columnas: {df.shape[1]}")
df.dtypes

In [ ]:
df.head(10)

## 2. Volumen de nulos por columna

`FTHG`/`FTAG` traen goles vacíos en algunas filas, y `Date` trae al menos una fecha irreconocible — esto se ve como texto no numérico en `FTHG`/`FTAG` porque el CSV crudo no fuerza tipos.

In [ ]:
columnas_clave = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG"]
nulos = df[columnas_clave].replace("", pd.NA).isna().sum()
porcentaje = (nulos / len(df) * 100).round(2)

reporte_nulos = pd.DataFrame({"nulos": nulos, "porcentaje": porcentaje})
reporte_nulos

In [ ]:
reporte_nulos["nulos"].plot(kind="bar", title="Valores vacíos por columna", color="#c0392b")
plt.ylabel("cantidad de filas")
plt.tight_layout()
plt.show()

## 3. Inconsistencia de nombres de equipo

Mismo equipo, dos formas de escribirlo en el CSV crudo — esto es exactamente el problema que resuelve `ml.data.alias` en el backend.

In [ ]:
equipos_crudos = pd.concat([df["HomeTeam"], df["AwayTeam"]])
equipos_crudos.value_counts()

## 4. Cruce contra la tabla de alias

`equipo_alias_demo.csv` es la misma tabla que usa `ml.data.alias.resolver_alias` en el backend. Acá se reimplementa el cruce en pandas puro (sin importar el paquete `ml`, para que el notebook corra también en Colab sin el resto del repo).

In [ ]:
alias = pd.read_csv("equipo_alias_demo.csv")
nombres_con_alias = set(alias["alias"])
nombres_del_csv = set(equipos_crudos.unique())

sin_alias = sorted(nombres_del_csv - nombres_con_alias)
print(f"Equipos distintos en el CSV: {len(nombres_del_csv)}")
print(f"Resueltos por equipo_alias_demo.csv: {len(nombres_del_csv) - len(sin_alias)} "
      f"({round(100 * (len(nombres_del_csv) - len(sin_alias)) / len(nombres_del_csv), 1)}%)")
print("Sin alias registrado (requieren agregar fila a equipo_alias_demo.csv):")
for nombre in sin_alias:
    print(f"  - {nombre}")

## 5. Distribución de goles

Forma de la variable que van a predecir Dixon-Coles y XGBoost.

In [ ]:
goles = pd.concat([
    pd.to_numeric(df["FTHG"], errors="coerce").dropna().rename("goles"),
    pd.to_numeric(df["FTAG"], errors="coerce").dropna().rename("goles"),
])
goles.plot(kind="hist", bins=range(0, 7), title="Distribución de goles por equipo y partido", color="#2980b9")
plt.xlabel("goles")
plt.tight_layout()
plt.show()

## 6. Partidos a lo largo del tiempo

Confirma que el CSV cubre un rango de fechas continuo (salvo la fila con fecha rota, que `pd.to_datetime` descarta con `errors="coerce"`).

In [ ]:
fechas = pd.to_datetime(df["Date"], format="%d/%m/%Y", errors="coerce")
print(f"Fechas irreconocibles: {fechas.isna().sum()}")
fechas.value_counts().sort_index().plot(kind="line", marker="o", title="Partidos por fecha")
plt.ylabel("partidos")
plt.tight_layout()
plt.show()

## 7. Para completar en vivo

Ideas para explorar en el momento de la defensa (a propósito sin resolver todavía):
- ¿Hay diferencia de goles entre local y visitante (ventaja de localía)?
- ¿Qué equipo tiene más partidos con datos faltantes?
- Repetir este notebook con el CSV real de Football-Data.co.uk una vez descargado.